# Phase 6 — 후속 질문과 문서별 대화 맥락

Phase 5 원문·인덱스를 그대로 사용합니다. 새 패키지를 설치하거나 인덱스를 재구축하지 않습니다.
기존 Phase 3에서 사용하던 `.venv/bin/python` 커널을 선택하고, pull 전에 열었던 커널은 Restart하세요.

첫 **Run All은 API 0회**입니다. 문서 ID 선택 → 예산 질문 → 기간 후속질문 → 비교 → 모호한 지칭 확인 → 문서 전환을 점검합니다.
이후 실제 생성은 마지막의 `RUN_LIVE_QA=True`로 별도 실행합니다.
선택한 공개 문서만 API 허용 목록에 넣습니다. `.env`의 키는 코드·출력·대화 상태에 저장하지 않습니다.


In [ ]:
from pathlib import Path
import json,sys
from IPython.display import display,Markdown

PROJECT_ROOT = None
if PROJECT_ROOT is None:
    candidates=[Path.cwd(),Path.cwd().parent,Path.cwd()/'rfp_phase5_local_20261111']
    ROOT=next((p.resolve() for p in candidates if (p/'rfp_phase6/session.py').is_file()),None)
else:ROOT=Path(PROJECT_ROOT).expanduser().resolve()
if ROOT is None:raise FileNotFoundError('PROJECT_ROOT에 Phase5 프로젝트 폴더를 지정하세요.')
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
print('Python:',sys.executable)
print('프로젝트:',ROOT)


## 1. 원문·인덱스와 전체 문서 목록

아래 목록은 모든 원본 ID를 보여 줍니다. 동일 원문의 사본 ID는 대표 ID로 연결됩니다.
`chat.list_documents('검색어', include_aliases=True)`로 사업명·기관·ID를 검색할 수 있습니다.
파일명과 기관명은 대상을 고르는 메타데이터이며 답변의 사실 근거는 원문 인용입니다.


In [ ]:
from rfp_phase5.runtime import bootstrap_phase5
from rfp_phase6.runtime import make_conversation

if 'app' in globals() and isinstance(app,dict) and app.get('index') is not None:app['index'].close()
app=bootstrap_phase5(ROOT)
settings=json.loads((ROOT/'configs/phase6.json').read_text(encoding='utf-8'))
chat=make_conversation(app,enabled=False,config=settings['conversation'],retrieval=settings['retrieval'])
catalog=chat.list_documents()
all_ids=chat.list_documents(include_aliases=True)
display(all_ids)
print('원본 ID:',len(all_ids),'/ 대표 문서:',len(catalog))
print('실효 API 활성화:',chat.llm.config.enabled)


## 2. 선택 문서와 표시 함수

`SELECTED_DOC_IDS`를 원하는 원본/대표 ID 목록으로 바꿀 수 있습니다.
기본값은 전체 목록의 앞 두 대표 문서로, 대화 흐름을 API 없이 점검하기 위한 선택입니다.
`ask()`는 모델에 전달하는 독립 질문과 실제 본문 검색어를 나누어 표시합니다.
사업명은 문서 범위를 정하는 데 사용하고, 본문은 질문 주제로 검색합니다.
생성되지 않은 경우 원문 후보를 앞 3개로 숨기지 않고 모두 표시합니다.


In [ ]:
SELECTED_DOC_IDS = [row['doc_id'] for row in catalog[:2]]
if not SELECTED_DOC_IDS:raise ValueError('로드된 문서가 없습니다.')

def show_result(result):
    route=result.get('conversation') or result
    display({key:route.get(key) for key in ['status','effective_doc_ids','filters','standalone_query','reason']})
    query_plan=result.get('retrieval',{}).get('content_query_plan') or route.get('retrieval_plan',{})
    if query_plan:
        display({'본문 검색어':query_plan.get('queries',[]),'검색 주제':query_plan.get('topics',[])})
    display(Markdown(result.get('markdown','')))
    candidates=result.get('candidate_documents',[])
    if candidates:
        display([{'번호':i,'doc_id':row['doc_id'],'사업명':row['project_title']}
                 for i,row in enumerate(candidates,1)])
    if not result.get('answer_generated') and result.get('source_chunks'):
        print('답변 입력에 포함된 원문 근거 후보:',len(result['source_chunks']),'개')
        display([{'문서':c['file_name'],'위치':c['locator'],'원문':c['text']}
                 for c in result['source_chunks']])
    print('상태:',result['status'],'/',result.get('execution_status'))
    return result

def ask(query,doc_ids=None):
    return show_result(chat.ask(query,**({'doc_ids':doc_ids} if doc_ids is not None else {})))

display([row for row in catalog if row['doc_id'] in SELECTED_DOC_IDS])


## 3. 같은 문서의 후속 질문

예산 질문 뒤의 기간 질문은 같은 문서를 유지합니다. 예산 질문의 숫자나 이전 답변을 기간 근거로 쓰지 않습니다.
매 턴 원문을 다시 검색하며, 검색 결과가 없으면 근거 미확보로 반환합니다.
‘몇 개월’은 ‘사업기간·과업기간·수행기간’으로 연결합니다. 기간 값 자체를 검색어에 넣지 않습니다.
비교 질문은 문서별 할당량 안에서 예산·기간 등 각 주제의 검색 결과를 함께 확보합니다.


In [ ]:
chat.reset()
budget=ask('사업예산은 얼마이고 부가세 포함인가요?',doc_ids=[SELECTED_DOC_IDS[0]])
period=ask('그건 몇 개월짜리야? 안정화도 그 안에 포함돼?')
assert period.get('effective_doc_ids')==[SELECTED_DOC_IDS[0]]
assert '직전 사용자 질문' not in period['conversation']['standalone_query']
assert chat.llm.usage_report()['api_calls']==0


## 4. 비교 뒤의 모호한 지칭과 문서 전환

두 문서를 비교한 뒤 “그건”이라고 하면 대상 확인을 요청합니다. 후보 번호 응답은 원래 질문을 이어서 처리합니다.
“첫 번째 사업/두 번째 사업”도 구별합니다. 문서 선택을 바꾸면 이전 사업의 질문 이력을 새 검색질문에 붙이지 않습니다.


In [ ]:
if len(SELECTED_DOC_IDS)>=2:
    compared=ask('두 문서의 사업예산과 사업기간을 비교해 줘.',doc_ids=SELECTED_DOC_IDS)
    ambiguous=ask('그건 몇 개월짜리야?')
    assert ambiguous['status']=='needs_clarification'
    resolved=ask('1번')
    assert resolved.get('effective_doc_ids')==[SELECTED_DOC_IDS[0]]
    switched=ask('제출기한과 제출방식은?',doc_ids=[SELECTED_DOC_IDS[1]])
    assert switched.get('effective_doc_ids')==[SELECTED_DOC_IDS[1]]
    assert len(chat.state['history'])==1
else:print('대표 문서가 한 개여서 비교·전환 예제는 생략합니다.')


## 5. 유지되는 필터와 대화 상태

지원 필터는 `agency`, `document_year`, `detected_format`입니다. 필터 밖 문서로 자동 전환하지 않습니다.
필터를 바꾸면 이전 주제를 지웁니다. `chat.set_filters({})`로 필터를 해제하고 `chat.reset()`으로 대화를 초기화합니다.
상태 파일은 사용자 질문·선택·필터만 담으며 이전 assistant 답변·API 키·원문은 포함하지 않습니다.
상태를 복원할 때 문서 ID·SHA를 포함한 목록 지문이 다르면 중단합니다.


In [ ]:
row=next(row for row in catalog if row['doc_id']==SELECTED_DOC_IDS[0])
chat.reset()
filter_values={k:row[k] for k in ('agency','document_year') if row.get(k) is not None}
chat.set_filters(filter_values)
filtered=ask('사업기간은?',doc_ids=[row['doc_id']])
assert filtered['filters']==filter_values

SAVE_USER_STATE = False
if SAVE_USER_STATE:
    state_path=chat.save(ROOT/'phase6_outputs/conversation_state.json')
    print('사용자 대화 상태:',state_path)
chat.reset()


## 6. 합성 회귀 테스트

실제 모델 답변을 대신하는 테스트 결과가 아닙니다. 독립적으로 만든 문서로 문서 혼입·잘못된 인용·모호한 지칭·상태 복원을 검사합니다.
반복되는 표지·양식 사이에 있는 기간 문장이 실제 검색 인덱스를 거쳐 생성 입력에 도달하는지도 검사합니다.


In [ ]:
import subprocess
checks=subprocess.run([sys.executable,'-m','unittest','rfp_phase6.test_session','rfp_phase6.test_retrieval','-q'],
                      cwd=ROOT,capture_output=True,text=True)
print(checks.stderr or checks.stdout)
assert checks.returncode==0,'Phase6 회귀 테스트에 실패했습니다.'
assert chat.llm.usage_report()['api_calls']==0
print('대화 경로·범위 점검: 통과 / 현재 대화의 API 호출: 0회')


## 7. 실제 OpenAI 답변 — 선택 실행

아래 `LIVE_APPROVED_DOC_IDS`의 문서를 확인한 뒤 `RUN_LIVE_QA=True`로 실행합니다.
`.env`는 프로젝트 폴더 또는 상위 폴더에서 읽으며 키 값을 출력하지 않습니다.
처음에는 동일 문서의 예산 → 기간 → 안정화 세 질문만 생성합니다. 현재 생성 클라이언트의 전체 호출 상한은6회입니다.
상한은 현재 프로세스/클라이언트 실행 내에서 누적되며 새 실행의 비용 한도나 청구 금액을 뜻하지 않습니다.
기간은 원문에 적힌 개월 수와 포함 조건을 확인합니다. 달력상 날짜 범위를 개월 수로 임의 환산하지 않습니다.
안정화가 원문에 명시되지 않은 사업은 포함 여부를 확정할 수 없다고 답할 수 있습니다.
각 결과와 실제 답변 입력은 로컬 `phase6_outputs/live_runs/`에 저장합니다.


In [ ]:
RUN_LIVE_QA = False
LIVE_APPROVED_DOC_IDS = SELECTED_DOC_IDS
DOTENV_PATH = None
live_results=[]

if type(RUN_LIVE_QA)is not bool:raise TypeError('RUN_LIVE_QA는 True/False여야 합니다.')
if RUN_LIVE_QA:
    from datetime import datetime,timezone
    import uuid
    from rfp_phase5_recovery.runner import load_api_key
    load_api_key(ROOT,DOTENV_PATH)
    live_chat=make_conversation(app,enabled=True,approved_doc_ids=LIVE_APPROVED_DOC_IDS,
        config=settings['conversation'],retrieval=settings['retrieval'],
        generation={**app['phase5_config']['generation'],'max_calls':6})
    live_chat.select_documents([LIVE_APPROVED_DOC_IDS[0]])
    run_dir=ROOT/'phase6_outputs/live_runs'/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ_')+uuid.uuid4().hex[:8])
    run_dir.mkdir(parents=True,exist_ok=False)
    print('실효 API 활성화:',live_chat.llm.config.enabled)
    for query in ['사업예산과 부가세 포함 여부를 알려줘.',
                  '그건 몇 개월짜리야?', '그 안에 안정화도 포함돼?']:
        live_results.append(show_result(live_chat.ask(query)))
        (run_dir/f'result_{len(live_results):03d}.json').write_text(
            json.dumps(live_results[-1],ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    display(live_chat.llm.usage_report())
    print('이번 실행 결과:',run_dir)
    chat=live_chat
    print('이후 ask()도 같은 실제 생성 세션과 남은 호출 상한을 사용합니다.')
else:print('실제 생성: 미실행 / 기본 Run All의 API 호출: 0회')

print('실제 생성 답변의 숫자·조건 정확성은 인용과 함께 확인합니다.')
print('문서 범위:',len(catalog),'개 대표 문서의 모든 ID 선택 가능')
print('Phase6 실행 기록:',{'offline_context_checks_passed':checks.returncode==0,
    'live_qa_executed':RUN_LIVE_QA,
    'live_answer_count':sum(bool(r.get('answer_generated')) for r in live_results),
    'live_statuses':[r.get('status') for r in live_results]})
